# 实验三：QQ邮箱采集（Playwright 版）

- 姓名：陈泓宇　学号：112411160433　班级：数据1241
- 运行位置：Docker 内 JupyterLab 的 Notebook 代码单元
- 说明：本 Notebook 按指导书四阶段 13 个单元补全；使用 Playwright 异步 API。
  - 第一阶段（单元1—2）：扫码登录
  - 第二阶段（单元3—6）：采集10封并保存 CSV
  - 第三阶段（单元7—9）：滚动采集与邮件ID去重
  - 第四阶段（单元10—12）：详情正文；单元13选做附件下载

> 补全时把每个 `____` 替换成正确的方法/参数。选择器基于 QQ 邮箱真实页面（参考第06课 Selenium 脚本中的 class 名），
> 建议在自己登录后的页面里用开发者工具二次核对后再运行。

## 第一阶段：扫码登录（第1—2课时）

### 单元1：准备浏览器

In [ ]:
import sys
from pathlib import Path
from uuid import uuid4
import pandas as pd
from playwright.async_api import async_playwright

PRIVATE = Path.cwd() / "private_data"
PRIVATE.mkdir(exist_ok=True)

# 使用 visual 安装时已有的显示服务配置。
sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))
import course_visual
course_visual.prepare()

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, args=["--disable-dev-shm-usage"])
context = await browser.new_context(
    viewport={"width": 1000, "height": 620}, accept_downloads=True
)
page = await context.new_page()
page.set_default_timeout(20000)  # 配置默认超时，单位毫秒，不需要await。

### 单元2：打开邮箱并等待扫码

In [ ]:
# ① 补打开网页的方法。
await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")
print("请在浏览器画面中扫码", flush=True)

# ② 人工扫码；自己查找左侧“收件箱”所在的区域。
inbox = page.locator("a").get_by_text("收件箱", exact=True)

# ③ 最长等待10分钟。出现后立即继续。
await inbox.wait_for(timeout=600000)
print("登录成功", flush=True)
await inbox.click()

## 第二阶段：采集10封邮件并保存（第3—4课时）

### 单元3：自己找选择器

只给两条线索，其余在开发者工具中查找。以下选择器基于 QQ 邮箱真实页面结构，建议登录后核验：

In [ ]:
ROW_CSS = ".mail-list-page-item"        # 一封普通邮件
SENDER_CSS = ".cmp-account-nick"        # 发件人姓名
AD_CSS = ".gg-unread-count"             # 广告聚合入口标记，应跳过
SUBJECT_CSS = ".mail-subject"           # 主题
TIME_CSS = ".mail-time"                 # 列表时间
HOVER_CSS = ".cmp-account-nick"         # 发件人悬停区域（触发邮箱卡片）
EMAIL_CSS = ".cmp-account-email"        # 卡片中的邮箱

### 单元4：定义列表解析函数

In [ ]:
async def get_ids():
    await page.locator(ROW_CSS).first.wait_for()
    # 一次读取整批ID，避免列表更新时逐个保存旧元素。
    ids = await page.locator(ROW_CSS).evaluate_all(
        "rows => rows.map(row => row.getAttribute('data-mailid'))"
    )
    if not ids or not all(ids):
        raise ValueError("没有取得有效邮件ID，请检查页面")
    return ids


def get_row(mail_id):
    return page.locator(f'{ROW_CSS}[data-mailid="{mail_id}"]')


async def read_list_mail(mail_id):
    row = get_row(mail_id)
    if await row.locator(AD_CSS).count():
        return None
    sender = await row.locator(SENDER_CSS).inner_text()
    subject = await row.locator(SUBJECT_CSS).inner_text()
    time_text = await row.locator(TIME_CSS).inner_text()

    # 先移开鼠标，等待上一张卡片消失。
    await row.locator(TIME_CSS).hover()
    await page.locator(EMAIL_CSS).wait_for(state="hidden")
    # 先把发件人滚入可见区域。滚动时页面会暂时停止响应悬停。
    await row.locator(HOVER_CSS).scroll_into_view_if_needed()
    await page.locator(".mail-list-page .ui-float-scroll-body-scrolling").wait_for(state="hidden")
    # 滚动结束后，再悬停本封发件人，等待卡片显示。
    await row.locator(HOVER_CSS).hover()
    await page.locator(EMAIL_CSS).wait_for(state="visible")
    email = await page.locator(EMAIL_CSS).inner_text()
    return {"邮件ID": mail_id, "发件人": sender,
            "邮箱": email, "主题": subject, "时间": time_text}

### 单元5：采集10封

In [ ]:
basic_records = []
for mail_id in await get_ids():
    record = await read_list_mail(mail_id)
    if record is None:
        continue
    basic_records.append(record)
    if len(basic_records) == 10:
        break
print("成功采集：", len(basic_records))

### 单元6：用pandas保存和回读

In [ ]:
def save_records(records, filename, columns):
    df = pd.DataFrame(records, columns=columns)
    path = PRIVATE / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def read_records(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)
basic_df = read_records(basic_path)
print("CSV记录数：", len(basic_df))
basic_df.head(10)

## 第三阶段：滚动采集与ID去重（第5—6课时）

### 单元7：补全解析和批次循环

In [ ]:
SCROLL_CSS = ".mail-list-page .ui-float-scroll-body-scrolling"  # 邮件列表滚动容器


def get_box():
    return page.locator(SCROLL_CSS)


async def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):
    result = []
    for mail_id in mail_ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:
            continue
        record = await read_list_mail(mail_id)
        if record is None:
            continue
        if include_details:
            # 第四阶段定义正文函数；本阶段False，不执行这里。
            record.update(await read_detail(mail_id))
        result.append(record)
        seen_ids.add(mail_id)
    return result


async def collect_batches(include_details=False, max_mails=0):
    await inbox.click()
    await page.locator(ROW_CSS).first.wait_for()
    await get_box().evaluate("box => box.scrollTop = 0")
    records, seen_ids = [], set()
    for batch in range(3):
        top = await get_box().evaluate("box => box.scrollTop")
        result = await parse_batch(await get_ids(), seen_ids, include_details, max_mails)
        records.extend(result)
        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")
        if batch == 2 or (max_mails and len(seen_ids) >= max_mails):
            break
        box = get_box()
        height = await box.evaluate("box => box.clientHeight")
        full_height = await box.evaluate("box => box.scrollHeight")
        next_top = min(top + height, full_height - height)
        if next_top <= top:
            print("已到列表末尾")
            break
        old_ids = await get_ids()
        await box.evaluate("(box, value) => box.scrollTop = value", next_top)
        # 在浏览器中比较新旧ID，改变后才继续。
        await page.wait_for_function("""data => {
            const ids = Array.from(document.querySelectorAll(data.selector),
                                   node => node.getAttribute('data-mailid'));
            return ids.length > 0 && JSON.stringify(ids) !== JSON.stringify(data.ids);
        }""", arg={"selector": ROW_CSS, "ids": old_ids})
    return records, seen_ids

### 单元8：执行并保存3批

In [ ]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids = await collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print("CSV记录数：", len(page_df))
print("唯一ID数：", page_df["邮件ID"].nunique())
print("重复ID数：", page_df["邮件ID"].duplicated().sum())

### 单元9：重复采集验证

In [ ]:
replay_ids = [r["邮件ID"] for r in page_records[:5]]
assert replay_ids, "先成功采集一些邮件，再做重复验证"
before = page_path.read_bytes()
replayed = await parse_batch(replay_ids, seen_ids)
print("重复采集新增：", len(replayed))
save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)
print("CSV内容未变：", before == page_path.read_bytes())

## 第四阶段：详情正文（第7—8课时）

### 单元10：补全正文函数

正文容器选择器 `BODY_CSS` 与“返回”控件文字 `BACK_TEXT` 需在自己登录后的页面里用开发者工具确认，下面给出 QQ 邮箱常用值供核对。

In [ ]:
BODY_CSS = ".mail-content-container .mail-content"   # 正文容器，请在页面中核验
BACK_TEXT = "返回"                                    # 返回控件的完整文字
DOWNLOAD_ATTACHMENTS = False


async def read_detail(mail_id):
    top = await get_box().evaluate("box => box.scrollTop")
    row = get_row(mail_id)
    await row.locator(SUBJECT_CSS).click()
    try:
        body_node = page.locator(BODY_CSS)
        await body_node.wait_for(state="attached")
        body = (await body_node.inner_text()).strip()
        files = ""
        if DOWNLOAD_ATTACHMENTS:
            files = await download_attachments(mail_id)
        return {"正文": body, "附件文件": files}
    finally:
        await page.get_by_text(BACK_TEXT, exact=True).click()
        await get_box().wait_for()
        await get_box().evaluate("(box, value) => box.scrollTop = value", top)
        await get_row(mail_id).wait_for(state="attached")

### 单元11：先试3封

In [ ]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]
trial_records, _ = await collect_batches(include_details=True, max_mails=3)
trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)
trial_df = read_records(trial_path)
print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))

### 单元12：正式采集3批正文

In [ ]:
detail_records, _ = await collect_batches(include_details=True)
detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)
detail_df = read_records(detail_path)
print("正文CSV记录数：", len(detail_df))
print("重复ID数：", detail_df["邮件ID"].duplicated().sum())

### 单元13：选做附件下载

不选做就跳过本单元，保持开关关闭。选做时补完框架后，只试3封，不覆盖正式正文CSV。

In [ ]:
ATTACH_CSS = ".att-item"        # 附件卡片，请在页面中核验
DOWNLOAD_TEXT = "下载"          # 下载控件文字


async def download_attachments(mail_id):
    cards = page.locator(ATTACH_CSS)
    if await cards.count() == 0:
        return ""
    # 每次建新目录，避免覆盖上次的附件。
    folder = PRIVATE / "attachments" / mail_id / uuid4().hex[:8]
    folder.mkdir(parents=True, exist_ok=False)
    files = []
    for number, card in enumerate(await cards.all(), 1):
        # 先监听下载，再点击。保存方法会等待下载完成。
        async with page.expect_download(timeout=90000) as event:
            await card.get_by_text(DOWNLOAD_TEXT, exact=True).click()
        download = await event.value
        name = Path(download.suggested_filename.replace("\\", "/")).name
        target = folder / f"{number:02d}_{name}"
        await download.save_as(target)
        files.append(str(target.relative_to(PRIVATE)))
    return "\n".join(files)


DOWNLOAD_ATTACHMENTS = True
attachment_records, _ = await collect_batches(include_details=True, max_mails=3)
attachment_path = save_records(attachment_records, "mail_attachments_trial.csv", DETAIL_COLUMNS)
print(read_records(attachment_path)[["邮件ID", "附件文件"]])
DOWNLOAD_ATTACHMENTS = False

## 结束：关闭浏览器

> 下课前保存 Notebook，再执行下面的关闭，最后运行 `stop-course-visual.bat`。

In [ ]:
await browser.close()
await p.stop()